# 19 — Bài báo, bước 2a: chiến lược D, TF-IDF + bộ phân loại tuyến tính (máy local, CPU)

**Phạm vi**

Baseline học máy truyền thống cho cả ba vị trí: tầng 1 (ACTIONABLE / NOT_ACTIONABLE), tầng 2 (BUG / FEATURE / UX) và phẳng
bốn lớp. Thiết kế, lưới và tiêu chí chọn đã pre-register ở amendment v3.2.

**Vai trò các tập dữ liệu**

Train dùng để huấn luyện, Validation dùng để chọn cấu hình và ngưỡng. Hold-out, focused và Lazada chỉ được đọc từ file text-only
để dự đoán sau khi cấu hình đã khóa; nhãn của ba tập này không được đọc trong notebook.

**Chấm điểm**

Không chấm ba tập khóa ở đây. Việc chấm được làm một lần trong notebook đánh giá chung của bước 2, sau khi mọi mô hình mới đã khóa.

In [1]:
import os, sys, json, datetime, subprocess, platform
import numpy as np, pandas as pd, yaml, sklearn
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification import evaluate as E
from src.classification.final_eval import load_frozen, predictions_long, parse_pm
from src.classification.learning_curve import tier1_metrics, tier2_metrics
from src.classification.tier1_actionable.mapping import to_tier1
from src.classification.tier1_actionable import threshold as TH
from src.classification.tier2_bfu.mapping import to_tier2, TIER2_LABELS
from src.classification.flat_baseline.mapping import to_flat
from src.classification.strategy_tfidf import model as D
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
SEEDS = load_frozen(ROOT)['tier1']['seeds']
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
train_df = pd.read_csv(os.path.join(DATA, 'shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(os.path.join(DATA, 'shopee_validation.csv'), encoding='utf-8-sig')
assert not set(train_df['review_id']) & set(val_df['review_id']), 'Train và Validation trùng review'
LOCKED_TEXT = {s: pd.read_csv(os.path.join(DATA, f'{s}_text_only.csv'), encoding='utf-8-sig') for s in ('holdout', 'focused', 'lazada')}
for s, d in LOCKED_TEXT.items():
    assert list(d.columns) == ['review_id', 'text'], f'{s}_text_only.csv có cột ngoài review_id, text'
print('Ngày chạy:', RUN_DATE, '| commit:', COMMIT, '| Python', sys.version.split()[0], '| scikit-learn', sklearn.__version__)
print('CPU:', platform.processor(), '| seeds:', SEEDS)
print('Train:', len(train_df), '| Validation:', len(val_df), '| tập khóa (text-only):', {s: len(d) for s, d in LOCKED_TEXT.items()})
print('Lưới:', len(D.GRID), 'cấu hình mỗi tầng =', D.FEATURES, '×', D.CLASSIFIERS, '× C', D.C_VALUES)

Ngày chạy: 15/11/2026 | commit: 2927f88af499d702a5bd8145977e8f06411aca57 | Python 3.13.1 | scikit-learn 1.9.1
CPU: Intel64 Family 6 Model 140 Stepping 1, GenuineIntel | seeds: [42, 1337, 2024]
Train: 1200 | Validation: 350 | tập khóa (text-only): {'holdout': 1000, 'focused': 150, 'lazada': 500}
Lưới: 18 cấu hình mỗi tầng = ('word', 'char', 'word+char') × ('logreg', 'linearsvc') × C (0.1, 1.0, 10.0)


## Tầng 1: dò lưới và chọn ngưỡng

**Cách chọn cấu hình**

Mỗi cấu hình train một lần với seed 42 trên Train 1.200 (gồm ambiguous) và được xếp theo Macro-F1 hai lớp ở argmax trên Validation.

**Ngưỡng**

Luật v2.0 áp trên xác suất ACTIONABLE của seed 42: trong các ngưỡng 0,01–0,90 có recall ACTIONABLE ≥ 0,80, lấy ngưỡng có
Macro-F1 cao nhất.

In [2]:
T1_TR, T1_VA = to_tier1(train_df), to_tier1(val_df)
G1, B1 = D.grid_search(T1_TR['text'], T1_TR['y'], T1_VA['text'], T1_VA['y'], tier1_metrics)
print(f"Tầng 1 — {len(G1)} cấu hình, Macro-F1 Validation (argmax) {G1['macro_f1'].min():.3f}–{G1['macro_f1'].max():.3f}")
print(G1.head(6).round(4).to_string(index=False))
print('Cấu hình thắng:', B1)
R1 = D.run_seeds(B1, T1_TR['text'], T1_TR['y'], {'validation': T1_VA['text']}, SEEDS)
SW1 = TH.sweep(T1_VA['y'], R1[42]['probs']['validation'][:, 1])
THR1 = TH.choose(SW1)
REC_MIN1 = min(TH.metrics_at(T1_VA['y'], R1[s]['probs']['validation'][:, 1], THR1['threshold'])['recall_actionable'] for s in SEEDS)
ROW1 = TH.summary_row(T1_VA['y'], [{'probs': R1[s]['probs']['validation']} for s in SEEDS], THR1, 'TF-IDF', B1)
print('\nNgưỡng chọn:', THR1)
print('Recall ACTIONABLE thấp nhất qua ba seed:', round(REC_MIN1, 3))
print(pd.DataFrame([ROW1]).to_string(index=False))

Tầng 1 — 18 cấu hình, Macro-F1 Validation (argmax) 0.759–0.823
 thứ tự lưới  features classifier    C  f1_actionable  recall_actionable  precision_actionable  macro_f1  giây train
           1      word     logreg  1.0         0.7183             0.7183                0.7183    0.8233       0.133
           2      word     logreg 10.0         0.7023             0.6479                0.7667    0.8169       0.161
          13 word+char     logreg  1.0         0.6912             0.6620                0.7231    0.8084       0.267
           0      word     logreg  0.1         0.6988             0.8169                0.6105    0.8026       0.136
           4      word  linearsvc  1.0         0.6769             0.6197                0.7458    0.8016       0.349
          12 word+char     logreg  0.1         0.6849             0.7042                0.6667    0.8009       0.245
Cấu hình thắng: {'features': 'word', 'classifier': 'logreg', 'C': 1.0}



Ngưỡng chọn: {'threshold': 0.45, 'recall_constraint_met': True, 'min_recall': 0.8, 'recall_actionable': 0.8028169014084507, 'precision_actionable': 0.6129032258064516, 'f1_actionable': 0.6951219512195121, 'macro_f1': 0.8009191845649799}
Recall ACTIONABLE thấp nhất qua ba seed: 0.803
strategy                                               cấu hình  ngưỡng  đạt recall ≥ 0.80 (seed 42)  recall thấp nhất (3 seed) f1_actionable recall_actionable precision_actionable   macro_f1
  TF-IDF {"features": "word", "classifier": "logreg", "C": 1.0}    0.45                         True                      0.803    69.5 ± 0.0        80.3 ± 0.0           61.3 ± 0.0 80.1 ± 0.0


## Tầng 2: dò lưới

**Dữ liệu**

Train actionable bỏ review ambiguous; đánh giá trên gold-actionable của Validation, giữ ambiguous.

**Cách chọn**

Mỗi cấu hình train một lần với seed 42 và được xếp theo Macro-F1 ba lớp. Cấu hình thắng dự đoán cả toàn bộ Validation để ghép cascade.

In [3]:
T2_TR, T2_VA = to_tier2(train_df, drop_ambiguous=True), to_tier2(val_df, drop_ambiguous=False)
print('Train tầng 2:', len(T2_TR), '| Validation gold-actionable:', len(T2_VA))
G2, B2 = D.grid_search(T2_TR['text'], T2_TR['y'], T2_VA['text'], T2_VA['y'], tier2_metrics)
print(f"Tầng 2 — {len(G2)} cấu hình, Macro-F1 Validation {G2['macro_f1'].min():.3f}–{G2['macro_f1'].max():.3f}")
print(G2.head(6).round(4).to_string(index=False))
print('Cấu hình thắng:', B2)
R2 = D.run_seeds(B2, T2_TR['text'], T2_TR['y'], {'validation_actionable': T2_VA['text'], 'validation': val_df['text']}, SEEDS)
ROW2 = E.summary_row(T2_VA['y'].to_numpy(), [{'probs': R2[s]['probs']['validation_actionable']} for s in SEEDS],
                     'TF-IDF', B2, 'Validation (gold-actionable)', TIER2_LABELS)
print(pd.DataFrame([ROW2]).to_string(index=False))

Train tầng 2: 321 | Validation gold-actionable: 71


Tầng 2 — 18 cấu hình, Macro-F1 Validation 0.688–0.809
 thứ tự lưới  features classifier    C  macro_f1  f1_BUG  f1_FEATURE  f1_UX  giây train
          12 word+char     logreg  0.1    0.8093  0.7778      0.7500 0.9000       0.129
          13 word+char     logreg  1.0    0.8057  0.8000      0.7391 0.8780       0.189
           0      word     logreg  0.1    0.7819  0.7500      0.6957 0.9000       0.035
           1      word     logreg  1.0    0.7797  0.7500      0.7111 0.8780       0.077
           7      char     logreg  1.0    0.7773  0.7719      0.6818 0.8780       0.112
           5      word  linearsvc 10.0    0.7762  0.7692      0.6957 0.8636       0.135
Cấu hình thắng: {'features': 'word+char', 'classifier': 'logreg', 'C': 0.1}


                         tập strategy                                                    cấu hình  n  n_BUG  n_FEATURE  n_UX   macro_f1     f1_BUG f1_FEATURE      f1_UX
Validation (gold-actionable)   TF-IDF {"C": 0.1, "classifier": "logreg", "features": "word+char"} 71     27         22    22 80.9 ± 0.0 77.8 ± 0.0 75.0 ± 0.0 90.0 ± 0.0


## Phẳng bốn lớp: dò lưới

**Cách chọn**

Train 1.200 với nhãn bốn lớp, gồm ambiguous; mỗi cấu hình train một lần với seed 42 và được xếp theo Macro-F1 bốn lớp trên Validation.

In [4]:
F_TR, F_VA = to_flat(train_df), to_flat(val_df)
GF, BF = D.grid_search(F_TR['text'], F_TR['y'], F_VA['text'], F_VA['y'], lambda y, p: E.metrics(y, p, E.FOUR_CLASSES))
print(f"Phẳng — {len(GF)} cấu hình, Macro-F1 Validation {GF['macro_f1'].min():.3f}–{GF['macro_f1'].max():.3f}")
print(GF[['thứ tự lưới', 'features', 'classifier', 'C', 'macro_f1', 'f1_BUG', 'f1_FEATURE', 'f1_UX', 'f1_OTHER']].head(6).round(4).to_string(index=False))
print('Cấu hình thắng:', BF)
RF = D.run_seeds(BF, F_TR['text'], F_TR['y'], {'validation': F_VA['text']}, SEEDS)
ROWF = E.summary_row(F_VA['y'].to_numpy(), [{'probs': RF[s]['probs']['validation']} for s in SEEDS],
                     'Phẳng TF-IDF', BF, 'Validation', E.FOUR_CLASSES)

Phẳng — 18 cấu hình, Macro-F1 Validation 0.550–0.665
 thứ tự lưới  features classifier    C  macro_f1  f1_BUG  f1_FEATURE  f1_UX  f1_OTHER
          13 word+char     logreg  1.0    0.6648  0.5098      0.4583 0.7692    0.9217
           7      char     logreg  1.0    0.6548  0.5231      0.4000 0.7895    0.9068
           4      word  linearsvc  1.0    0.6548  0.4286      0.4615 0.8000    0.9292
           5      word  linearsvc 10.0    0.6512  0.4000      0.4737 0.8000    0.9313
          12 word+char     logreg  0.1    0.6462  0.4474      0.4286 0.8205    0.8885
           8      char     logreg 10.0    0.6426  0.4906      0.3556 0.8108    0.9133
Cấu hình thắng: {'features': 'word+char', 'classifier': 'logreg', 'C': 1.0}


## So sánh trên Validation

**Cascade TF-IDF**

Tầng 1 ở ngưỡng đã chọn ghép với tầng 2 cùng seed: review bị gọi NOT_ACTIONABLE nhận nhãn OTHER, còn lại nhận nhãn tầng 2.

**Tham chiếu**

Số của ViSoBERT, Bi-LSTM + fastText và LLM lấy từ các file Validation đã có (`gd10_gate_validation.csv`,
`llm_validation_metrics.csv`, `cascade_vs_flat_validation.csv`). Validation chỉ có 71 review actionable nên chênh lệch nhỏ chưa có ý nghĩa.

In [5]:
Y4V = F_VA['y'].to_numpy()
CAS = [E.cascade_labels(R1[s]['probs']['validation'][:, 1] >= THR1['threshold'], R2[s]['probs']['validation'].argmax(1)) for s in SEEDS]
ROWC = E.summary_row(Y4V, [{'pred': p} for p in CAS], 'Cascade TF-IDF', f"tầng 1 ngưỡng {THR1['threshold']} + tầng 2", 'Validation', E.FOUR_CLASSES)
GATE = pd.read_csv(os.path.join(RES, 'gd10_gate_validation.csv')).set_index('model')
LLMV = pd.read_csv(os.path.join(RES, 'llm_validation_metrics.csv')).set_index('tầng')
CVF = pd.read_csv(os.path.join(RES, 'cascade_vs_flat_validation.csv')).dropna(subset=['tập']).set_index('strategy')
g = lambda k: f"{GATE.loc[k, 'new_mean']:.1f} ± {GATE.loc[k, 'new_std']:.1f}"
VAL = pd.DataFrame([
    {'chỉ số': 'Tầng 1 Macro-F1 (2 lớp, ở ngưỡng)', 'TF-IDF': ROW1['macro_f1'], 'Bi-LSTM + fastText': g('Tầng 1 – Bi-LSTM + fastText (ngưỡng 0.09)'),
     'ViSoBERT': g('Tầng 1 – ViSoBERT (ngưỡng 0.11)'), 'LLM few-shot': f"{LLMV.loc['tier1', 'macro_f1'] * 100:.1f}"},
    {'chỉ số': 'Tầng 1 Recall ACTIONABLE', 'TF-IDF': ROW1['recall_actionable'], 'Bi-LSTM + fastText': '', 'ViSoBERT': '',
     'LLM few-shot': f"{LLMV.loc['tier1', 'recall_actionable'] * 100:.1f}"},
    {'chỉ số': 'Tầng 2 Macro-F1 (3 lớp)', 'TF-IDF': ROW2['macro_f1'], 'Bi-LSTM + fastText': g('Tầng 2 – Bi-LSTM + fastText'),
     'ViSoBERT': g('Tầng 2 – ViSoBERT'), 'LLM few-shot': f"{LLMV.loc['tier2', 'macro_f1'] * 100:.1f}"},
    {'chỉ số': 'End-to-end cascade Macro-F1 (4 lớp)', 'TF-IDF': ROWC['macro_f1'], 'Bi-LSTM + fastText': '',
     'ViSoBERT': CVF.loc['Cascade ViSoBERT', 'macro_f1'], 'LLM few-shot': ''},
    {'chỉ số': 'End-to-end phẳng Macro-F1 (4 lớp)', 'TF-IDF': ROWF['macro_f1'], 'Bi-LSTM + fastText': '',
     'ViSoBERT': CVF.loc['Baseline phẳng ViSoBERT', 'macro_f1'], 'LLM few-shot': ''}])
print('Validation (%; ô trống là số không có sẵn trong file Validation đã lưu):'); print(VAL.to_string(index=False))
print(); print(pd.DataFrame([ROWC, ROWF]).to_string(index=False))

Validation (%; ô trống là số không có sẵn trong file Validation đã lưu):
                             chỉ số     TF-IDF Bi-LSTM + fastText   ViSoBERT LLM few-shot
  Tầng 1 Macro-F1 (2 lớp, ở ngưỡng) 80.1 ± 0.0         78.6 ± 3.9 85.0 ± 0.6         89.0
           Tầng 1 Recall ACTIONABLE 80.3 ± 0.0                                       87.3
            Tầng 2 Macro-F1 (3 lớp) 80.9 ± 0.0         72.0 ± 3.2 75.6 ± 2.2         79.9
End-to-end cascade Macro-F1 (4 lớp) 67.2 ± 0.0                    68.8 ± 0.8             
  End-to-end phẳng Macro-F1 (4 lớp) 66.5 ± 0.0                    69.1 ± 2.0             

       tập       strategy                                                    cấu hình   n  n_BUG  n_FEATURE  n_UX  n_OTHER   macro_f1     f1_BUG f1_FEATURE      f1_UX   f1_OTHER
Validation Cascade TF-IDF                                 tầng 1 ngưỡng 0.45 + tầng 2 350     27         22    22      279 67.2 ± 0.0 48.5 ± 0.0 44.8 ± 0.0 85.0 ± 0.0 90.7 ± 0.0
Validation   Phẳng TF-IDF {"C"

## Khóa cấu hình

**Nội dung khóa**

Cấu hình thắng của ba tầng, ngưỡng tầng 1, seed, định nghĩa đặc trưng và phiên bản thư viện được ghi vào
`03_configs/dev/paper_tfidf_dev_config.yaml` rồi khóa thành `03_configs/frozen/paper_tfidf_config.yaml`.

**Không ghi đè**

Nếu file khóa đã tồn tại với nội dung khác, notebook dừng; thay đổi cấu hình đã khóa cần amendment mới.

In [6]:
CONTENT = {'tfidf': {
    'seeds': SEEDS, 'canonical_seed': 42, 'amendment': 'v3.2',
    'preprocessing': 'Unicode NFC + chữ thường', 'features': D.FEATURE_SPEC,
    'grid': {'features': list(D.FEATURES), 'classifier': list(D.CLASSIFIERS), 'C': list(D.C_VALUES),
             'class_weight': 'balanced', 'linearsvc': 'CalibratedClassifierCV(method=sigmoid, cv=5)'},
    'tier1': {'config': B1, 'threshold': THR1['threshold'],
              'threshold_rule': {'min_recall': 0.8, 'sweep': [0.01, 0.9, 0.01], 'chosen_on_seed': 42, 'amendment': 'v2.0'},
              'recall_constraint_met': bool(THR1['recall_constraint_met']), 'recall_min_over_seeds_validation': round(float(REC_MIN1), 3),
              'data': 'Train 1.200 (gồm ambiguous); ACTIONABLE = BUG ∪ FEATURE ∪ UX'},
    'tier2': {'config': B2, 'decision': 'argmax', 'data': 'Train actionable, bỏ ambiguous khi train'},
    'flat': {'config': BF, 'decision': 'argmax', 'data': 'Train 1.200, nhãn 4 lớp, gồm ambiguous'},
    'library': f'scikit-learn {sklearn.__version__}', 'python': sys.version.split()[0]}}
body = yaml.safe_dump(CONTENT, allow_unicode=True, sort_keys=False)
DEV = os.path.join(ROOT, '03_configs', 'dev', 'paper_tfidf_dev_config.yaml')
FROZEN = os.path.join(ROOT, '03_configs', 'frozen', 'paper_tfidf_config.yaml')
open(DEV, 'w', encoding='utf-8').write(f'# DEV — chiến lược D (TF-IDF), chọn trên Validation ngày {RUN_DATE}, amendment v3.2.\n' + body)
head = f'# FROZEN — khóa ngày {RUN_DATE}, amendment v3.2. Không sửa sau mốc này; mọi thay đổi phải có amendment mới.\n'
if os.path.exists(FROZEN):
    old = open(FROZEN, encoding='utf-8').read()
    assert old.split('\n', 1)[1] == body, 'File khóa đã tồn tại với nội dung khác — dừng, cần amendment mới.'
    print('File khóa đã có và trùng nội dung:', os.path.relpath(FROZEN, ROOT))
else:
    open(FROZEN, 'w', encoding='utf-8').write(head + body)
    print('Đã khóa:', os.path.relpath(FROZEN, ROOT))
print(body)

Đã khóa: 03_configs\frozen\paper_tfidf_config.yaml
tfidf:
  seeds:
  - 42
  - 1337
  - 2024
  canonical_seed: 42
  amendment: v3.2
  preprocessing: Unicode NFC + chữ thường
  features:
    word: TfidfVectorizer(analyzer='word', ngram_range=(1, 2), token_pattern=r'(?u)\b\w+\b',
      sublinear_tf=True)
    char: TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)
  grid:
    features:
    - word
    - char
    - word+char
    classifier:
    - logreg
    - linearsvc
    C:
    - 0.1
    - 1.0
    - 10.0
    class_weight: balanced
    linearsvc: CalibratedClassifierCV(method=sigmoid, cv=5)
  tier1:
    config:
      features: word
      classifier: logreg
      C: 1.0
    threshold: 0.45
    threshold_rule:
      min_recall: 0.8
      sweep:
      - 0.01
      - 0.9
      - 0.01
      chosen_on_seed: 42
      amendment: v2.0
    recall_constraint_met: true
    recall_min_over_seeds_validation: 0.803
    data: Train 1.200 (gồm ambiguous); ACTIONABLE = BUG 

## Dự đoán các tập khóa

**Cách chạy**

Đọc lại cấu hình từ file khóa, train lại từng seed rồi dự đoán Validation và ba tập khóa từ file text-only. Xác suất Validation
phải trùng lần chạy chọn cấu hình, nếu không notebook dừng.

**Chi phí**

Thời gian train và thời gian suy luận cho 1.000 review (đo trên 1.000 review Hold-out text-only) được ghi theo cùng cột với
`gd10_costs_neural.csv`.

In [7]:
FZT = yaml.safe_load(open(FROZEN, encoding='utf-8'))['tfidf']
SETS = {'validation': val_df['text'], **{s: d['text'] for s, d in LOCKED_TEXT.items()}}
TRAIN = {'tier1': (FZT['tier1']['config'], T1_TR), 'tier2': (FZT['tier2']['config'], T2_TR), 'flat': (FZT['flat']['config'], F_TR)}
REF = {'tier1': {s: R1[s]['probs']['validation'] for s in SEEDS}, 'tier2': {s: R2[s]['probs']['validation'] for s in SEEDS},
       'flat': {s: RF[s]['probs']['validation'] for s in SEEDS}}
RUNS, COST = {}, []
for tier, (cfg, tr) in TRAIN.items():
    out = D.run_seeds(cfg, tr['text'], tr['y'], SETS, SEEDS, timing_set='holdout')
    for s in SEEDS:
        diff = float(np.abs(out[s]['probs']['validation'] - REF[tier][s]).max())
        assert diff < 1e-9, f'{tier}/seed {s}: xác suất Validation lệch {diff}'
        RUNS[(tier, 'tfidf', s)] = out[s]
    COST.append({'tier': tier, 'model': 'tfidf', 'n_train': len(tr),
                 'train_seconds_mean': float(np.mean([out[s]['train_seconds'] for s in SEEDS])),
                 'infer_seconds_per_1000_mean': float(np.mean([out[s]['infer_seconds_per_1000'] for s in SEEDS])),
                 'peak_gpu_gb_max': 0.0, 'gpu': f'không (CPU {platform.processor()})'})
COST = pd.DataFrame(COST)
print('Xác suất Validation trùng lần chạy chọn cấu hình ở cả ba tầng và ba seed.')
print(COST.round(4).to_string(index=False))
SEED_SAME = {tier: all(np.array_equal(RUNS[(tier, 'tfidf', s)]['probs']['holdout'], RUNS[(tier, 'tfidf', 42)]['probs']['holdout']) for s in SEEDS)
             for tier in TRAIN}
print('Ba seed cho dự đoán Hold-out giống hệt nhau:', SEED_SAME)

Xác suất Validation trùng lần chạy chọn cấu hình ở cả ba tầng và ba seed.
 tier model  n_train  train_seconds_mean  infer_seconds_per_1000_mean  peak_gpu_gb_max                                                             gpu
tier1 tfidf     1200              0.1227                       0.0255              0.0 không (CPU Intel64 Family 6 Model 140 Stepping 1, GenuineIntel)
tier2 tfidf      321              0.2526                       0.1391              0.0 không (CPU Intel64 Family 6 Model 140 Stepping 1, GenuineIntel)
 flat tfidf     1200              0.6362                       0.1131              0.0 không (CPU Intel64 Family 6 Model 140 Stepping 1, GenuineIntel)
Ba seed cho dự đoán Hold-out giống hệt nhau: {'tier1': True, 'tier2': True, 'flat': True}


## Lưu kết quả

**Các file**

Lưới, quét ngưỡng, bảng Validation, dự đoán Validation kèm nhãn, chi phí và dự đoán của ba tập khóa (chỉ xác suất, không nhãn)
được lưu với tiền tố `paper_tfidf_` và `paper_pred_tfidf_` trong `05_experiments/rq1_classification/`.

In [8]:
def save(df, name):
    df.to_csv(os.path.join(RES, f'{name}.csv'), index=False, encoding='utf-8-sig')

save(pd.concat([G1.assign(tầng='tầng 1'), G2.assign(tầng='tầng 2'), GF.assign(tầng='phẳng')], ignore_index=True), 'paper_tfidf_grid')
save(SW1, 'paper_tfidf_tier1_threshold_sweep')
save(pd.DataFrame([ROW1]), 'paper_tfidf_validation_tier1')
save(pd.DataFrame([ROW2, ROWC, ROWF]), 'paper_tfidf_validation')
save(VAL, 'paper_tfidf_validation_compare')
vp = predictions_long(val_df['review_id'], RUNS, 'validation').merge(val_df[['review_id', 'gold_label']], on='review_id')
save(vp, 'paper_tfidf_val_predictions')
save(COST, 'paper_tfidf_costs')
for s, d in LOCKED_TEXT.items():
    save(predictions_long(d['review_id'], RUNS, s), f'paper_pred_tfidf_{s}')
json.dump({'run_date': RUN_DATE, 'commit': COMMIT, 'python': sys.version.split()[0], 'sklearn': sklearn.__version__,
           'cpu': platform.processor(), 'amendment': 'v3.2', 'seed_identical_predictions': SEED_SAME},
          open(os.path.join(RES, 'paper_tfidf_run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if f.startswith(('paper_tfidf_', 'paper_pred_tfidf_'))))

Đã lưu: ['paper_pred_tfidf_focused.csv', 'paper_pred_tfidf_holdout.csv', 'paper_pred_tfidf_lazada.csv', 'paper_tfidf_costs.csv', 'paper_tfidf_grid.csv', 'paper_tfidf_run_info.json', 'paper_tfidf_tier1_threshold_sweep.csv', 'paper_tfidf_val_predictions.csv', 'paper_tfidf_validation.csv', 'paper_tfidf_validation_compare.csv', 'paper_tfidf_validation_tier1.csv']


## Nhận xét

**Cấu hình thắng**

Logistic Regression thắng ở cả ba tầng. Tầng 1 dùng word 1–2-gram với C = 1 (Macro-F1 0,823 ở argmax, lưới 0,759–0,823), tầng 2
dùng word+char với C = 0,1 (0,809, lưới 0,688–0,809) và phẳng dùng word+char với C = 1 (0,665, lưới 0,550–0,665). Cấu hình
tầng 2 nằm ở mép dưới của C; lưới không được mở rộng, cùng nguyên tắc đã áp cho tầng 2 của ViSoBERT và Bi-LSTM.

**Ngưỡng tầng 1**

Ngưỡng 0,45 cho recall ACTIONABLE 0,803 trên Validation, vừa qua sàn 0,80 của luật v2.0.

**So với các chiến lược khác trên Validation**

Macro-F1 tầng 1 đạt 80,1, nằm giữa Bi-LSTM + fastText (78,6 ± 3,9) và ViSoBERT (85,0 ± 0,6). Macro-F1 tầng 2 đạt 80,9, cao hơn
ViSoBERT (75,6 ± 2,2) và LLM (79,9), nhưng Validation chỉ có 71 review actionable và lưới tầng 2 của hai mô hình neural dừng ở mép,
nên chênh lệch này chưa đủ để kết luận. End-to-end, cascade TF-IDF đạt 67,2 và phẳng TF-IDF đạt 66,5, thấp hơn ViSoBERT lần lượt
1,6 và 2,6 điểm.

**Tính tất định**

Ba seed cho dự đoán giống hệt nhau vì Logistic Regression giải bằng lbfgs không phụ thuộc seed. Độ lệch chuẩn bằng 0 phản ánh
điều này, không phải độ ổn định đo được qua các lần chạy.

**Chi phí**

Mỗi tầng train dưới 1 giây và suy luận 0,03–0,14 giây cho 1.000 review trên CPU, không cần GPU hay API. Số nhãn cần giống ViSoBERT.

**Bước tiếp theo**

Hold-out, focused và Lazada đã có dự đoán nhưng chưa được chấm; việc chấm diễn ra một lần trong notebook đánh giá chung của bước 2.